# Notebook 02 — Normalización 0-1 y tuning con grid search

Aplicamos `MinMaxScaler` 0-1 a los features y exploramos un grid de hiperparámetros con validación cruzada 3-fold. El hallazgo metodológico clave es el **gap CV/test**: el mejor CV resulta peor en test que los hiperparámetros defaults del artículo, lo que indica que la CV aleatoria sobreajusta a la estructura espacial. Esto motiva el Notebook 04 (CV espacial).

## Compatibilidad con Google Colab

La siguiente celda detecta si el notebook se ejecuta en Colab y, en ese caso, instala `gpboost` y `shap` y clona el repositorio para tener acceso a los datos derivados (`data/`) y a los modelos entrenados (`modelos/`). En Jupyter local no hace nada.

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print('Entorno Colab — instalando dependencias...')
    !pip -q install gpboost shap
    if not os.path.isdir('/content/repo'):
        !git clone -q https://github.com/linaq11/examen-2-gpboost.git /content/repo
    os.chdir('/content/repo')
    print('CWD:', os.getcwd())
else:
    # En local: asume que el notebook está en repo/notebooks/
    here = os.path.dirname(os.path.abspath('__file__')) if '__file__' in globals() else os.getcwd()
    if os.path.basename(here) == 'notebooks':
        os.chdir(os.path.dirname(here))
    print('CWD:', os.getcwd())

In [ ]:
import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gpboost as gpb

np.random.seed(1)
DATA = 'data'
MOD  = 'modelos'
FIG  = 'figuras'
os.makedirs(FIG, exist_ok=True)
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import KFold

## 1. Datos y normalización 0-1

In [ ]:
df = pd.read_csv(os.path.join(DATA, 'simulado_train_seed1.csv'))
X      = df[['X1','X2']].values
y      = df['y'].values
coords = df[['s_1','s_2']].values

scaler_X = MinMaxScaler().fit(X)
scaler_c = MinMaxScaler().fit(coords)
X_n      = scaler_X.transform(X)
coords_n = scaler_c.transform(coords)

with open(os.path.join(MOD, 'scalers_minmax.json'), 'w') as f:
    json.dump({'X_min': scaler_X.data_min_.tolist(),
               'X_max': scaler_X.data_max_.tolist(),
               'coords_min': scaler_c.data_min_.tolist(),
               'coords_max': scaler_c.data_max_.tolist()}, f, indent=2)
print('Scalers persistidos. Rango X normalizado:', X_n.min(), '–', X_n.max())

## 2. Grid search + CV 3-fold

Grid pequeño (36 combinaciones) sobre `learning_rate`, `max_depth`, `min_data_in_leaf` y `num_boost_round`. Cada combinación entrena un GPBoost completo con early stopping.

In [ ]:
grid = {
    'learning_rate': [0.005, 0.01, 0.05],
    'max_depth':     [3, 5, 7],
    'min_data_in_leaf': [5, 10, 20],
    'num_boost_round': [247, 500]
}
from itertools import product
combos = list(product(*grid.values()))
print(f'Combinaciones: {len(combos)}  (~38 s estimados con CV 3-fold)')

In [ ]:
# El grid search ya se ejecutó y los resultados están en data/tuning_resultados.json
# Aquí solo cargamos y mostramos la conclusión.
with open(os.path.join(DATA, 'tuning_resultados.json')) as f:
    tuning = json.load(f)
print(json.dumps(tuning, indent=2))

## 3. Hallazgo: gap CV/test

El JSON anterior resume tres grids progresivamente más exhaustivos (8, 12 y 36 combinaciones). En los tres, la combinación con mejor RMSE de CV resulta *peor* en el test grid 50×50 que los defaults del artículo:

| Grid | mejor-CV RMSE | test RMSE de ese params | defaults test RMSE |
|---|---|---|---|
| 8 combinaciones | 0.32 | 0.6816 | **0.6073** |
| 12 combinaciones | 0.36 | 0.6706 | **0.6073** |
| 36 combinaciones | 0.43 | 0.6619 | **0.6073** |

Es decir, gap CV vs test $\approx 0{,}30$ unidades de RMSE. La causa es que los folds aleatorios comparten vecinos espaciales con el entrenamiento, así que el modelo aprende patrones que se replican en folds pero no en el grid completo. **Decisión:** se conservan los defaults de Sigrist 2021 como modelo final.

In [ ]:
# Defaults del artículo (modelo final), tal cual el Notebook 01:
gp = gpb.GPModel(gp_coords=coords, cov_function='exponential')
params_default = {'objective':'regression_l2','learning_rate':0.01,'max_depth':3,
                  'min_data_in_leaf':10,'num_leaves':1024,'verbose':-1}
ds = gpb.Dataset(X, y)
bst_def = gpb.train(params=params_default, train_set=ds, gp_model=gp,
                    num_boost_round=247)
print('Modelo final (defaults Sigrist 2021) reentrenado y disponible en memoria.')

### Lectura

La validación cruzada aleatoria selecciona combinaciones que *sobreajustan* a los folds aleatorios porque cada fold contiene puntos espacialmente cercanos a su entrenamiento. El test grid 50×50 (que cubre todo el dominio sin agruparse alrededor de los puntos de entrenamiento) penaliza esos hiperparámetros. La consecuencia práctica es: **para datos espaciales, validar con CV aleatoria es engañoso**. El Notebook 04 hace explícita esta crítica con CV por bloques.